# PS-S06E02: Model Stacking


This notebook tackles the [**Playground Series – Season 6, Episode 2: Predicting Heart Disease**](https://www.kaggle.com/competitions/playground-series-s6e2), a competition focused on predicting if a patient has heart disease, based on features describing the patient's demographics and selected health metrics.

We combine multiple base models (e.g., XGBoost, neural nets) using their **out-of-fold (OOF) probabilities** as features for a second-level meta-model.

Evaluation metric: **ROC AUC**.


## Ensemble Strategy: Stacking with a Logistic Meta-Model

Blending (weighted averaging) can fail when one model is dominant or when models are highly correlated. Stacking treats ensembling as a supervised learning problem: we train a small meta-model to combine base model probabilities.

A practical advantage is that a weaker model can still help if it is *differently wrong*—the meta-model can learn when to trust it.


## Install Needed Packages

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV

from ps_s06e02_experiment_setup import ExperimentSetup

warnings.filterwarnings('ignore')

%matplotlib inline


In [2]:
helper = ExperimentSetup()

seed = helper.set_seeds()

helper.configure_pandas()
helper.suppress_warnings()

TARGET = 'Heart Disease'


Random seed set to: 10301
Warnings suppressed.


In [3]:
# Load ground truth labels
training_df = helper.read_dataset('training')

y_true = training_df[TARGET].map({'Absence': 0, 'Presence': 1}).astype(np.int8)

TRAINING DATASET

   id  Age  Sex  Chest pain type   BP  Cholesterol  FBS over 120  EKG results  \
0   0   58    1                4  152          239             0            0   
1   1   52    1                1  125          325             0            2   
2   2   56    0                2  160          188             0            2   
3   3   44    0                3  134          229             0            2   
4   4   58    1                4  140          234             0            2   

   Max HR  Exercise angina  ST depression  Slope of ST  \
0     158                1          3.600            2   
1     171                0          0.000            1   
2     151                0          0.000            1   
3     150                0          1.000            2   
4     125                1          3.800            2   

   Number of vessels fluro  Thallium Heart Disease  
0                        2         7      Presence  
1                        0         3    

In [4]:
submission_df = helper.read_dataset('submission')

## Loading OOF and Test Probabilities

We load the Out-of-Fold (OOF) **probability estimates** and the Test-set **probability estimates** produced by individual model notebooks.

* **OOF probabilities** are used to train and evaluate the stacker. Because they come from validation folds, they are an unbiased proxy for generalization.
* **Test probabilities** are the inputs the trained meta-model will combine to produce the final submission.


In [5]:
oof_files = []
test_files = []

if helper.running_in_kaggle():
    # Expect a dataset like ps-s06e02-<something> that contains predictions/s06e02
    pred_dir = '/kaggle/input/ps-s06e02-*/predictions'
else:
    pred_dir = 'predictions'

# Fallback if you kept a flat predictions/ directory
if not glob.glob(pred_dir):
    pred_dir = 'predictions'

oof_files = sorted(glob.glob(f'{pred_dir}/*_oof_probs.csv'))
test_files = sorted(glob.glob(f'{pred_dir}/*_test_probs.csv'))

print(f'Found {len(oof_files)} OOF files and {len(test_files)} Test files.')
print('OOF files:')
for f in oof_files:
    print(' -', os.path.basename(f))
print('Test files:')
for f in test_files:
    print(' -', os.path.basename(f))


Found 4 OOF files and 4 Test files.
OOF files:
 - catboost_oof_probs.csv
 - lgb_oof_probs.csv
 - nn-tabular-resnet_oof_probs.csv
 - xgb_oof_probs.csv
Test files:
 - catboost_test_probs.csv
 - lgb_test_probs.csv
 - nn-tabular-resnet_test_probs.csv
 - xgb_test_probs.csv


In [6]:
# Helper to load and merge probabilities
def load_probs(file_list, index_col='id'):
    series_list = []
    for file in file_list:
        model_name = os.path.basename(file).replace('_oof_probs.csv', '').replace('_test_probs.csv', '')
        df = pd.read_csv(file)
        # Probability column = first column that isn't id/target
        prob_col = [c for c in df.columns if c not in ['id', 'target']][0]
        s = df.rename(columns={prob_col: model_name}).set_index(index_col)[model_name]
        series_list.append(s)
    return pd.concat(series_list, axis=1)


In [7]:
# Create DataFrames
oof_df = load_probs(oof_files)
test_df = load_probs(test_files)

# Look for missing rows in test data
missing_by_model = test_df.isna().sum().sort_values(ascending=False)
print("Missing test rows per model:")
print(missing_by_model[missing_by_model > 0])

if missing_by_model.any():
    bad = missing_by_model[missing_by_model > 0].index.tolist()
    print("\nFirst few missing ids for each bad model:")
    for m in bad:
        print(m, test_df.index[test_df[m].isna()][:10].tolist())

# Create an id-indexed label series (0/1)
y_by_id = pd.Series(y_true.values, index=training_df['id'].values)

# Keep only rows where all models have OOF probabilities
oof_df = oof_df.sort_index().dropna(axis=0)
y_true_aligned = y_by_id.loc[oof_df.index].values

# Align test rows by id as well
test_df = test_df.sort_index()

print(f'OOF Shape: {oof_df.shape}')
print(f'Test Shape: {test_df.shape}')


Missing test rows per model:
Series([], dtype: int64)
OOF Shape: (630000, 4)
Test Shape: (270000, 4)


In [8]:
print('\nIndividual Model ROC AUC (OOF):')
individual_scores = {}
for model in oof_df.columns:
    auc = roc_auc_score(y_true_aligned, oof_df[model].values)
    individual_scores[model] = auc
    print(f'{model}: {auc:.6f}')

best_single_model = max(individual_scores, key=individual_scores.get)
best_single_auc = individual_scores[best_single_model]
print(f'\nBest model/AUC: {best_single_model}/{best_single_auc:.6f}')



Individual Model ROC AUC (OOF):
catboost: 0.955556
lgb: 0.955485
nn-tabular-resnet: 0.953233
xgb: 0.955453

Best model/AUC: catboost/0.955556


## Stacking Optimization: Logistic Regression Meta-Model

We train a second-level meta-model on the base models' **OOF probabilities**.

* **Features (X):** OOF probabilities from each base model.
* **Target (y):** true label (0=Absence, 1=Presence).

We use **logistic regression with L2 regularization** as the meta-model. It is fast, stable under correlation, and outputs a valid probability in [0, 1].
The regularization strength is tuned via internal cross-validation.


In [9]:
# ==========================================
# STACKING STRATEGY (Meta-Model)
# ==========================================

print('Preparing stacking data...')

model_cols = list(oof_df.columns)
print('Stacking models:', model_cols)

X_stack = oof_df[model_cols].values
y_stack = y_true_aligned
X_test_stack = test_df[model_cols].values

# Standardize meta-features (helps logistic regression)
scaler = StandardScaler()
X_stack_s = scaler.fit_transform(X_stack)
X_test_stack_s = scaler.transform(X_test_stack)

# LogisticRegressionCV tunes regularization strength for ROC AUC
meta_model = LogisticRegressionCV(
    Cs=20,
    cv=5,
    scoring='roc_auc',
    penalty='l2',
    solver='lbfgs',
    max_iter=5000,
    n_jobs=-1,
    random_state=seed,
)

meta_model.fit(X_stack_s, y_stack)

oof_meta_prob = meta_model.predict_proba(X_stack_s)[:, 1]
auc_meta = roc_auc_score(y_stack, oof_meta_prob)
print(f'Meta-model OOF ROC AUC: {auc_meta:.6f}')


Preparing stacking data...
Stacking models: ['catboost', 'lgb', 'nn-tabular-resnet', 'xgb']
Meta-model OOF ROC AUC: 0.955536


## Final Ensemble & Submission

We generate test-set probabilities by applying the trained meta-model to the matrix of base-model test probabilities.
The submission file uses the competition's sample submission schema.


In [10]:
# Generate test probabilities from the stacker
test_meta_prob = meta_model.predict_proba(X_test_stack_s)[:, 1]

# Fill submission using sample_submission column name
sub = submission_df.copy()
target_col = [c for c in sub.columns if c != 'id'][0]
sub[target_col] = test_meta_prob
sub.to_csv('submission.csv', index=False)
print('Saved: submission.csv')

print('\nSUBMISSION')
print('==========')
print(sub.head(10))


Saved: submission.csv

SUBMISSION
       id  Heart Disease
0  630000          0.947
1  630001          0.037
2  630002          0.958
3  630003          0.036
4  630004          0.136
5  630005          0.956
6  630006          0.037
7  630007          0.677
8  630008          0.958
9  630009          0.038
